In [3]:
import numpy as np
import sys
!{sys.executable} -m pip install pandas
import pandas as pd


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [9]:
df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol'],
    'score': [98, 89, 85],
    'grade': ['A', 'B', 'C']
})

df[['score','name']]

,score,name
0,98,Alice
1,89,Bob
2,85,Carol


In [10]:
df.loc[0]

name     Alice
score       98
grade        A
Name: 0, dtype: object

In [12]:
df.loc[0:1, ['name', 'score']]

,name,score
0,Alice,98
1,Bob,89


In [13]:
df.iloc[0]

name     Alice
score       98
grade        A
Name: 0, dtype: object

In [18]:
# df[df['score'] > 80]
# df[(df['score'] > 70) & (df['grade'] == 'B')]
df.query("score > 80 and grade == 'A'")


,name,score,grade
0,Alice,98,A


In [22]:
df['pass'] = df['score'] > 80
df

,name,score,grade,pass
0,Alice,98,A,True
1,Bob,89,B,True
2,Carol,85,C,True


In [23]:
df['score_norm'] = df['score'].apply(lambda x: (x-50)/50)
df

,name,score,grade,pass,score_norm
0,Alice,98,A,True,0.96
1,Bob,89,B,True,0.78
2,Carol,85,C,True,0.70


In [24]:
df.rename(columns={'score': 'marks'}, inplace=True)
df

,name,marks,grade,pass,score_norm
0,Alice,98,A,True,0.96
1,Bob,89,B,True,0.78
2,Carol,85,C,True,0.70


In [25]:
df.drop(index=0)
df

,name,marks,grade,pass,score_norm
0,Alice,98,A,True,0.96
1,Bob,89,B,True,0.78
2,Carol,85,C,True,0.70


In [30]:
# df.set_index('name', inplace=True)
df.reset_index(inplace=True)
df

,name,marks,grade,pass,score_norm
0,Alice,98,A,True,0.96
1,Bob,89,B,True,0.78
2,Carol,85,C,True,0.70


groupby follows the Split → Apply → Combine pattern: split the data into groups, apply a function to each group independently, then combine the results into a new structure.

In [11]:
sales = pd.DataFrame({
    'region': ['North', 'South', 'North', 'South', 'North'],
    'product': ['A', 'A', 'B', 'B', 'A'],
    'revenue': [200, 150, 300, 250, 180],
    'units':   [20, 15, 30, 25, 18]
})
sales


,region,product,revenue,units
0,North,A,200,20
1,South,A,150,15
2,North,B,300,30
3,South,B,250,25
4,North,A,180,18


In [9]:
print(sales.groupby('region')['revenue'].sum())


region
North    680
South    400
Name: revenue, dtype: int64


In [10]:
print(sales.groupby('region').mean(numeric_only=True))

           revenue      units
region                       
North   226.666667  22.666667
South   200.000000  20.000000


In [12]:
print(sales.groupby('region')['revenue'].transform('sum'))

0    680
1    400
2    680
3    400
4    680
Name: revenue, dtype: int64


In [13]:
# Normalise each row by its group total
sales['rev_pct'] = (
    sales['revenue'] /
    sales.groupby('region')['revenue'].transform('sum')
)
# Z-score within each region
sales['rev_z'] = sales.groupby('region')['revenue'].transform(
    lambda x: (x - x.mean()) / x.std()
)

In [14]:
print(sales.groupby('region').filter(lambda g: g['revenue'].sum() > 500))

  region product  revenue  units   rev_pct     rev_z
0  North       A      200     20  0.294118 -0.414781
2  North       B      300     30  0.441176  1.140647
4  North       A      180     18  0.264706 -0.725866


In [15]:
employees = pd.DataFrame({
    'emp_id':  [1, 2, 3, 4],
    'name':    ['Alice', 'Bob', 'Carol', 'Dave'],
    'dept_id': [10, 20, 10, 30]
})

departments = pd.DataFrame({
    'dept_id': [10, 20],
    'dept':    ['Engineering', 'Marketing']
})

In [16]:
# INNER join (default) — only matching rows
pd.merge(employees, departments, on='dept_id')

,emp_id,name,dept_id,dept
0,1,Alice,10,Engineering
1,2,Bob,20,Marketing
2,3,Carol,10,Engineering


In [17]:
# LEFT join  — all employees, NaN if no dept match
pd.merge(employees, departments, on='dept_id', how='left')

,emp_id,name,dept_id,dept
0,1,Alice,10,Engineering
1,2,Bob,20,Marketing
2,3,Carol,10,Engineering
3,4,Dave,30,NaN


In [19]:
pd.merge(employees, departments, on='dept_id', how='right')

,emp_id,name,dept_id,dept
0,1,Alice,10,Engineering
1,3,Carol,10,Engineering
2,2,Bob,20,Marketing


In [20]:
pd.merge(employees, departments, on='dept_id', how='outer')

,emp_id,name,dept_id,dept
0,1,Alice,10,Engineering
1,3,Carol,10,Engineering
2,2,Bob,20,Marketing
3,4,Dave,30,NaN


In [25]:
# Both DataFrames must share a common index
a = pd.DataFrame({'A': [1, 2]}, index=['x', 'y'])
b = pd.DataFrame({'B': [3, 4]}, index=['x', 'y'])

a.join(b)             # left join by default
a.join(b, how='inner')

,A,B
x,1,3
y,2,4


In [33]:
# Stack rows (axis=0, default)
combined = pd.concat([a, b], ignore_index=True)
combined

,A,B
0,1.0,NaN
1,2.0,NaN
2,NaN,3.0
3,NaN,4.0


In [34]:
combined = pd.concat([a, b], axis=1, ignore_index=True)
combined

,0,1
x,1,3
y,2,4


In [31]:
print(pd.concat([a,b], keys=['2023', '2024']))

          A    B
2023 x  1.0  NaN
     y  2.0  NaN
2024 x  NaN  3.0
     y  NaN  4.0
